# Proteomics integration with an enzyme-constrained model

In [ ]:
# Setup. On Google Colab this downloads the course files and installs the dependencies (~2 min).
# Locally (conda env from environment.yml) it only moves to the course folder so `data/` is found.
import os, sys

if "google.colab" in sys.modules:
    if not os.path.isdir("/content/cell-factory-design-course"):
        !git clone -q --depth 1 https://github.com/lab-biotek-bio-ugm/cell-factory-design-course /content/cell-factory-design-course
        %pip install -q --progress-bar off -r /content/cell-factory-design-course/requirements.txt
    from google.colab import output
    output.enable_custom_widget_manager()  # needed for Escher maps

for folder in (".", "..", "/content/cell-factory-design-course"):
    if os.path.isdir(os.path.join(folder, "data")):
        os.chdir(folder)
        break

## Overview

**Learning objectives.** After this notebook you should be able to:

1. Explain how an enzyme-constrained model links reaction fluxes to enzyme amounts through turnover numbers ($k_{cat}$).
2. Convert absolute proteomics data (protein copies per cell) into model units (mmol gDW$^{-1}$).
3. Constrain an enzyme-constrained model with measured enzyme levels and explain why it may fail to reach the measured growth rate.
4. Use shadow prices to relax ("flexibilize") the most limiting measurements and identify enzymatic bottlenecks.
5. Compare predictions (exchange fluxes, flux ranges) of models with and without proteomics constraints.

**Estimated time.** 45 min for the walk-through, 30 min for the exercises.

**Prerequisites.** Notebooks 02 (model content), 04 (flux variability analysis) and 05 (changing the medium); basic pandas.

### Background

**Enzyme-constrained models.** Classic FBA only uses stoichiometry and flux bounds, so it has no notion of the protein a cell must invest to run a pathway. The GECKO method ([Sánchez et al., 2017](https://doi.org/10.15252/msb.20167411)) adds enzymes to a genome-scale model. Every enzyme becomes a pseudo-metabolite that is consumed by the reaction it catalyses with coefficient $1/k_{cat}$, which enforces

$$
v_j \le k_{cat,j}\,[E_j]
$$

with $v_j$ in mmol gDW$^{-1}$ h$^{-1}$, $k_{cat}$ in h$^{-1}$ and the enzyme concentration $[E_j]$ in mmol gDW$^{-1}$. To make these constraints work:

* reversible reactions are split into a forward and a backward (`_REV`) reaction, because enzyme usage must be positive in both directions (this also applies to exchange reactions: uptake happens through `EX_..._e_REV`);
* reactions catalysed by several isoenzymes are split into one copy per isoenzyme (suffix `No1`, `No2`, ...);
* each enzyme `prot_<UniProt ID>` is supplied by a pseudo-exchange reaction `prot_<UniProt ID>_exchange`. Its upper bound is the amount of enzyme available.

The model used here is eciML1515, the GECKO version of the _E. coli_ K-12 MG1655 model iML1515 ([Monk et al., 2017](https://doi.org/10.1038/nbt.3956)). [VERIFY: which GECKO/ecModels release this file comes from; the SBML only carries the id `eciML1515_v1.0` and no provenance annotation.] The enzyme exchanges in this file are unbounded, so without data the model behaves like a normal model with extra bookkeeping.

**Proteomics data.** We use absolute protein abundances of _E. coli_ BW25113 grown in 22 conditions ([Schmidt et al., 2016](https://doi.org/10.1038/nbt.3418)), reported as copies per cell with a coefficient of variation (CV). The main part uses growth on **acetate**. To turn copies per cell into model units:

$$
\frac{\text{mmol}}{\text{gDW}} = \frac{\text{copies/cell}\times 10^{3} / N_A}{V_\text{cell}\,\rho\,f_\text{dry}}
$$

with Avogadro's number $N_A$, the measured cell volume $V_\text{cell}$ (fL), cell density $\rho \approx 1.105$ g mL$^{-1}$ $= 1.105\times10^{-12}$ g fL$^{-1}$ and dry mass fraction $f_\text{dry} \approx 0.3$ gDW g$^{-1}$. [VERIFY: sources for $\rho = 1.105$ g/mL and $f_\text{dry} = 0.3$; both are common textbook approximations for _E. coli_.]

**Flexibilization.** Measurements are noisy, $k_{cat}$ values are often taken from other organisms or conditions, and not every protein is fully active. Constraining the model with all measurements at once therefore usually makes it grow much slower than measured. A practical fix is to relax measurements one by one: solve the model, pick the enzyme whose constraint has the largest effect on growth (its **shadow price**), remove its upper bound, and repeat until the model reaches the measured growth rate. The relaxed enzymes are candidates for measurement errors or wrong $k_{cat}$ values, and the enzymes that remain limiting are the **bottlenecks** of the proteome-constrained model.

Load the enzyme-constrained model of _Escherichia coli_. The helper `use_dual_simplex` works around a GLPK performance problem with this model (see its docstring).

In [ ]:
import pandas as pd
import swiglpk

from cobra.io import read_sbml_model
from cobra.flux_analysis import flux_variability_analysis



def use_dual_simplex(model):
    """Switch GLPK to the dual simplex method.

    GLPK's default (primal) simplex can stall for minutes on this large, badly scaled
    model, while the dual simplex solves it in well under a second. The setting is not
    kept by `model.copy()`, so call this again for every copy.
    """
    model.solver.configuration._smcp.meth = swiglpk.GLP_DUALP


model = read_sbml_model("data/eciML1515.xml.gz")
use_dual_simplex(model)

The model has two differences with a standard COBRA model. First, the reactions contain an extra _metabolite_: the enzyme itself. Fumarate reductase (FRD2) consumes `prot_P00363` (FrdA) and its other subunits with a coefficient of $1/k_{cat}$ (in h, since $k_{cat}$ is in h$^{-1}$).

In [ ]:
model.reactions.FRD2No1

In this model, all protein ids follow the form `prot_<UniProt ID>`.

The second difference is the existence of _protein exchange reactions_, named `prot_<UniProt ID>_exchange`. They supply the enzyme and are unbounded in this model.

In [ ]:
model.reactions.prot_P00363_exchange

By putting an upper bound on these exchanges we integrate proteomics data into the model. After that it can be simulated like any other COBRA model.

## 1. Proteomics data preparation

Proteomics data are usually reported as number of copies per cell. We need to convert them into the units of the enzyme-constrained model (mmol gDW$^{-1}$). The data files are supplementary tables of the proteome study ([Schmidt et al., 2016](https://doi.org/10.1038/nbt.3418)) exported to TSV: S5 contains the protein copies per cell for 22 conditions and S23 the growth rates and cell volumes.

In [ ]:
df = pd.read_csv(
    #"data/ecoli_proteomics_schmidt2016_S9.tsv", # our strain but just Glc/LB
    "data/ecoli_proteomics_schmidt2016_S5_ren.tsv", # BW25113 22 media
    sep="\t", skiprows=2  # skip titles and subtitles (XLXS)
)
exp_details = pd.read_csv(
    "data/ecoli_details_schmidt2016_S23.tsv", sep="\t",
    skiprows=2  # skip titles and subtitles (XLXS)
)

In [ ]:
df.head()

In [ ]:
exp_details.head()

In [ ]:
df = df.loc[:, 
    df.columns.str.contains("_copies|_cv", regex=True) |  # only interested in copies/cell and uncertainty
    df.columns.isin(["Uniprot Accession"])  # and relevant info about proteins
]

In [ ]:
df_ac = df.loc[:, ["Uniprot Accession", "Acetate_copies", "Acetate_cv"]]
# rename resulting columns
df_ac.columns = ["uniprot", "copies_per_cell", "CV"]

In [ ]:
df_ac.describe()

In [ ]:
# allow for measurement uncertainty: extend the upper bound by half a standard deviation (CV is in %)
df_ac["copies_upper"] = df_ac["copies_per_cell"] + 0.5 * df_ac["CV"]/100 * df_ac["copies_per_cell"]

First, convert copies per cell to amount per cell, using Avogadro's number $N_A = 6.022\times10^{23}$ molecules mol$^{-1}$:

\begin{align}
\frac{\text{mmol}}{\text{cell}} = \frac{\text{molecules}}{\text{cell}} \cdot \frac{1}{N_A}\frac{\text{mol}}{\text{molecules}} \cdot 10^3\frac{\text{mmol}}{\text{mol}}
\end{align}

In [ ]:
df_ac["mmol_per_cell"] = df_ac["copies_upper"] * 1e3/6.022e23

Then convert the amount per cell into amount per gram dry weight. The dry weight of one cell is its volume times its density times the dry mass fraction:

\begin{align}
\frac{\text{mmol}}{\text{gDW}} = \frac{\text{mmol}}{\text{cell}} \cdot \frac{1}{V_\text{cell}\,[\text{fL}] \cdot \rho\,[\text{g/fL}] \cdot f_\text{dry}\,[\text{gDW/g}]}
\end{align}

The growth rate and cell volume for BW25113 on acetate are read from the experiment table.

In [ ]:
growth_experimental = exp_details.loc[
    (exp_details["Growth condition"]=="Acetate") & (exp_details["Strain"]=="BW25113"), 
    "Growth rate (h-1)"
].values[0]
cell_volume = exp_details.loc[
    (exp_details["Growth condition"]=="Acetate") & (exp_details["Strain"]=="BW25113"), 
    "Single cell volume [fl]1"
].values[0]
cell_density = 1.105e-12  # g/fL (= 1.105 g/mL)
dry_mass_fraction = 0.3  # gDW per g of cell mass (about 70 % of a cell is water)

In [ ]:
df_ac["conc"] = df_ac["mmol_per_cell"] * 1 / (cell_volume * cell_density * dry_mass_fraction)

In [ ]:
proteomics = df_ac["conc"]
proteomics.index = df_ac["uniprot"]

## 2. Constraining the model with proteomics

The proteome was measured during growth on acetate, so the model has to simulate acetate minimal medium as well (the file's default medium is glucose). Because exchanges are split, uptake runs through the `_REV` reactions. We leave acetate uptake unbounded: in the enzyme-constrained model the enzymes, not a fixed bound, should limit uptake.

In [ ]:
model.reactions.EX_glc__D_e_REV.upper_bound = 0  # no glucose
model.reactions.EX_ac_e_REV.upper_bound = 1000  # acetate uptake, limited only by the enzymes

In [ ]:
# save a copy without proteomics constraints for comparison
plain_model = model.copy()
use_dual_simplex(plain_model)

In [ ]:
def limit_proteins(model, measurements):
    """Apply proteomics measurements to `model`.

    Adapted from https://github.com/DD-DeCaF/simulations/blob/devel/src/simulations/modeling/driven.py

    Parameters
    ----------
    model: cobra.Model
        The enzyme-constrained model.
    measurements : pd.DataFrame
        Protein abundances in mmol / gDW.

    """
    for protein_id, measure in measurements.items():
        try:
            rxn = model.reactions.get_by_id(f"prot_{protein_id}_exchange")
        except KeyError:
            pass
        else:
            # update only upper_bound (as enzymes can be unsaturated):
            rxn.bounds = (0, measure)

Apply the measured enzyme amounts (acetate condition) and optimize the enzyme-constrained model.

In [ ]:
limit_proteins(model, proteomics)

In [ ]:
# enzyme-constrained: compare with growth_experimental

model.optimize()

The optimization is **infeasible**: with all measured enzyme amounts applied, the model cannot even carry the non-growth ATP maintenance flux (`ATPM` has a lower bound of 6.86 mmol gDW$^{-1}$ h$^{-1}$; relaxing it gives a growth rate of about 0.0002 h$^{-1}$). The real strain grows at about 0.3 h$^{-1}$ on acetate (`growth_experimental`), so some measured enzyme amounts are too low to carry the required fluxes. Note that 58 proteins were measured with 0 copies per cell.

## 3. Flexibilization

Experimental measurements can be too restrictive, in particular when their uncertainty is ignored or the $k_{cat}$ values in the model are too low. Flexibilization relaxes them iteratively:

1. solve the model and read the shadow prices of the protein pseudo-metabolites;
2. take the protein with the most negative shadow price, i.e. the one whose limited availability costs the most growth;
3. remove its measurement (set its exchange upper bound to 1000) and re-solve;
4. repeat until the growth rate reaches 1.05 × the measured one, or no measured proteins are left.

In [ ]:
def top_shadow_prices(solution, met_ids, top=1):
    """
    Retrieves shadow prices for a list of metabolites from the solution and ranks
    them from most to least sensitive in the model.

    Parameters
    ----------
    solution: cobra.Solution
        The usual Solution object returned by model.optimize().
    met_ids: iterable of strings
        Subset of metabolite IDs from the model.
    top: int
        The number of metabolites to be returned.

    Returns
    -------
    shadow_pr: pd.Series
        Top shadow prices, ranked.
    """
    shadow_pr = solution.shadow_prices
    shadow_pr = shadow_pr.loc[shadow_pr.index.isin(met_ids)]
    return shadow_pr.sort_values()[:top]


def flexibilize_proteomics(
    model, biomass_reaction, minimal_growth, proteomics
):
    """
    Replace proteomics measurements with a set that enables the model to grow. Proteins
    are removed from the set iteratively based on sensitivity analysis (shadow prices).
    
    Adapted from https://github.com/DD-DeCaF/simulations/blob/devel/src/simulations/modeling/driven.py

    Parameters
    ----------
    model: cobra.Model
        The enzyme-constrained model.
    minimal_growth_rate: float
        Minimal growth rate to enforce.
    proteomics: pandas.DataFrame
        List of measurements.

    Returns
    -------
    growth_rate: dict
        New growth rate (will change if the model couldn't grow at the inputted value).
    proteomics: list(dict)
        Filtered list of proteomics.

    """
    def protein_to_metabolite(protein_id, model):
        met_id = model.metabolites.query(lambda m: protein_id in m.id)
        return met_id[0].id if met_id else ""
    
    # reset growth rate in model:
    model.reactions.get_by_id(biomass_reaction).bounds = (0, 1000)

    # build a table with protein ids, met ids in model and values to constrain with:
    prot_df = pd.DataFrame(proteomics)
    prot_df.index = prot_df.index.astype("str")
    prot_df["met_id"] = [protein_to_metabolite(prot, model) for prot in prot_df.index]
    prot_df = prot_df[prot_df.met_id != ""]
    
    # constrain the model with all proteins and optimize:
    limit_proteins(model, proteomics)
    solution = model.optimize()
    new_growth_rate = solution.objective_value if solution.objective_value else 0
    
    # relax growth constraint
    minimal_growth *= 1.05

    # while the model cannot grow to the desired level, remove the protein with
    # the highest shadow price:
    prots_to_remove = []
    while new_growth_rate < minimal_growth and not prot_df.empty:
        # get most influential protein in model:
        top_protein = top_shadow_prices(solution, list(prot_df["met_id"]))
        top_protein = top_protein.index[0]
        top_protein = prot_df.index[prot_df["met_id"] == top_protein][0]

        # update data: append protein to list, remove from current dataframe and
        # increase the corresponding upper bound to +1000:
        prots_to_remove.append(top_protein)
        prot_df = prot_df.drop(labels=top_protein)
        limit_proteins(model, pd.Series(data=[1000], index=[top_protein]))

        # re-compute solution:
        solution = model.optimize()
        #if solution.objective_value == new_growth_rate:  # the algorithm is stuck
        #    break
        new_growth_rate = solution.objective_value if solution.objective_value else 0

    # update growth rate if optimization was not successful:
    if new_growth_rate < minimal_growth:
        print(
            f"Minimal growth was not reached! "
            f"Final growth of the model: {new_growth_rate}"
        )

    return new_growth_rate, prots_to_remove

Run the flexibilization with the measured growth rate on acetate as target. This takes up to a minute (one LP per removed protein).

In [ ]:
biomass_reaction = "BIOMASS_Ec_iML1515_core_75p37M"
new_growth_rate, prots_removed = flexibilize_proteomics(model, biomass_reaction, growth_experimental, proteomics)

In [ ]:
print(f"Proteins in dataset: {proteomics.shape[0]}\nProteins removed: {len(prots_removed)}")

In [ ]:
model.optimize()

In [ ]:
plain_model.optimize()

Let's compare the predictions of both models. First the active exchange reactions. In this model all fluxes are non-negative: uptakes appear as `EX_..._e_REV` reactions and secretions as `EX_..._e`.

Note that the model without proteomics has no limit on acetate uptake, so it predicts an unrealistically high growth rate (about 8 h$^{-1}$): without enzyme constraints you would need a measured uptake rate as input. It even takes up acetate at the maximum while secreting part of it again, an artefact of splitting reversible exchanges into two reactions: without enzyme costs nothing prevents running both directions at once.

In [ ]:
plain_exchanges = {r.id: r.flux for r in plain_model.exchanges if r.flux > 1e-9}
enzyme_exchanges = {r.id: r.flux for r in model.exchanges if r.flux > 1e-9}
pd.DataFrame({"without proteomics": plain_exchanges, "with proteomics": enzyme_exchanges}).fillna(0).sort_index()

Next, flux variability analysis of the exchange reactions at optimal growth. For the enzyme-constrained model this runs in a single process, so the dual simplex setting is used; it takes about half a minute.

In [ ]:
plain_fva = flux_variability_analysis(plain_model, reaction_list=plain_model.exchanges)

In [ ]:
enzyme_fva = flux_variability_analysis(model, reaction_list=model.exchanges, processes=1)

Exchange reactions with bounded ranges in the enzyme-constrained model. The ranges are narrow: the proteome leaves little freedom.

In [ ]:
enzyme_fva[enzyme_fva.maximum < 700].sort_values("maximum", ascending=False)

The same for the model without proteomics. Its ranges are much wider because nothing limits acetate uptake:

In [ ]:
plain_fva[plain_fva.maximum < 700].sort_values("maximum", ascending=False)

Finally, enzyme usage in the optimal solution of the enzyme-constrained model: how much of each measured enzyme is used (flux through its `prot_..._exchange`, in mmol gDW$^{-1}$) compared with the measured amount (its upper bound). Enzymes at 100 % saturation are fully used.

In [ ]:
usage = pd.DataFrame(
    {"used": r.flux, "measured": r.upper_bound}
    for r in model.reactions if r.id.startswith("prot_") and r.id.endswith("_exchange") and r.upper_bound < 1000
)
usage.index = [r.id for r in model.reactions if r.id.startswith("prot_") and r.id.endswith("_exchange") and r.upper_bound < 1000]
usage["saturation"] = usage["used"] / usage["measured"]
usage.sort_values("saturation", ascending=False).head(15)

## Exercises (30 min)

1. Identify the enzymatic bottlenecks of the enzyme-constrained model on **acetate** (hint: shadow prices of the protein pseudo-metabolites).
2. Prepare a model for growth on **glucose** and constrain it with the glucose proteomics data.
3. Does the model reach the measured growth rate? If not, flexibilize it.
4. Identify the enzymatic bottlenecks on **glucose** and compare them with acetate.

In [ ]:
# Your code here


## References

- Monk, Lloyd, Brunk et al. (2017). iML1515, a knowledgebase that computes Escherichia coli traits. *Nature Biotechnology* 35:904–908. [doi.org/10.1038/nbt.3956](https://doi.org/10.1038/nbt.3956)
- Schmidt, Kochanowski, Vedelaar et al. (2016). The quantitative and condition-dependent Escherichia coli proteome. *Nature Biotechnology* 34:104–110. [doi.org/10.1038/nbt.3418](https://doi.org/10.1038/nbt.3418)
- Sánchez, Zhang, Nilsson et al. (2017). Improving the phenotype predictions of a yeast genome‐scale metabolic model by incorporating enzymatic constraints. *Molecular Systems Biology* 13:935. [doi.org/10.15252/msb.20167411](https://doi.org/10.15252/msb.20167411)